In [8]:
import os
import zipfile
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Locate the ZIP file automatically in the current directory
zip_files = [f for f in os.listdir() if f.endswith('.zip')]

if zip_files:
    zip_path = zip_files[0]
    print(f"Found zip file: '{zip_path}'")
else:
    zip_path = input("Enter the full path to your .zip file: ").strip("'\"")

# 2. Extract and load dataset directly from ZIP
with zipfile.ZipFile(zip_path, 'r') as z:
    file_names = z.namelist()
    data_file = [f for f in file_names if f.endswith(('.csv', '.xlsx', '.xls'))][0]
    print(f"Loading dataset file from archive: '{data_file}'")
    
    if data_file.endswith('.csv'):
        df = pd.read_csv(z.open(data_file))
    else:
        df = pd.read_excel(z.open(data_file))

print(f"Dataset successfully loaded with shape: {df.shape}")

# 3. Clean Data & Select Target Column
cols_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 
                'Lat Long', 'Latitude', 'Longitude', 'Churn Reason']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

# Handle missing values explicitly without inplace=True (fixes the FutureWarning)
if 'Total Charges' in df.columns:
    df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
    df['Total Charges'] = df['Total Charges'].fillna(df['Total Charges'].median())

# Detect and format target column
target_col = 'Churn Label' if 'Churn Label' in df.columns else 'Churn'
X = df.drop(columns=[target_col, 'Churn Value', 'Churn Score'], errors='ignore')
y = df[target_col].map({'Yes': 1, 'No': 0}) if df[target_col].dtype == object else df[target_col]

# 4. Preprocessing Setup
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# 5. Build Model Pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    ))
])

# 6. Train-Test Split & Model Fitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model_pipeline.fit(X_train, y_train)

# 7. Model Evaluation
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

Found zip file: 'Telco_customer_churn.xlsx.zip'
Loading dataset file from archive: 'Telco_customer_churn.xlsx'
Dataset successfully loaded with shape: (7043, 33)

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.54      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.79      1409

ROC-AUC Score: 0.8508
